In [481]:
# Imports

import scipy
from pathlib import Path
import numpy as np
import pandas as pd
import trimesh
from dataclasses import dataclass
import numpy as np
from enum import Enum
import torch

In [482]:
# Code to generate a CSV overview of the dataset

# DATASET_ROOT = Path("dropbox")
# SUBFOLDERS = [
#     "boxes", "labels", "models", "obbs", "ops", "part mesh indices", "syms"
# ]

# rows = []

# for category_path in DATASET_ROOT.iterdir():
#     if not category_path.is_dir():
#         continue
#     category_name = category_path.name
#     syms_folder = category_path / "syms"
#     if not syms_folder.exists():
#         continue

#     for mat_file in syms_folder.glob("*.mat"):
#         data = scipy.io.loadmat(mat_file)
#         if "shapename" not in data:
#             continue
#         shape_name = str(data["shapename"].squeeze())
#         row_name = f"{category_name}_{mat_file.stem}"
#         row = {"shape": row_name}

#         for sub in SUBFOLDERS:
#             sub_path = category_path / sub
#             if sub_path.exists():
#                 if sub == "models":
#                     obj_file = sub_path / f"{shape_name}.obj"
#                     row[sub] = str(obj_file) if obj_file.exists() else None
#                 elif sub == "obbs":
#                     obb_file = sub_path / f"{shape_name}.obb"
#                     row[sub] = str(obb_file) if obb_file.exists() else None
#                 else:
#                     candidate = sub_path / mat_file.name
#                     row[sub] = str(candidate) if candidate.exists() else None
#             else:
#                 row[sub] = None

#         rows.append(row)

# df = pd.DataFrame(rows).set_index("shape").sort_index()
# df.columns = [c.replace(" ", "_") for c in df.columns]
# df.to_csv("dataset_overview.csv")
# print("Saved to 'dataset_overview.csv'")

In [483]:
# ShapeData class to load and store shape data

@dataclass
class ShapeData:
    def __init__(self, row):
        """Initialize ShapeData from a DataFrame row like chair_1_row."""

        # Load .mat files
        self.boxes = scipy.io.loadmat(row.boxes)["box"]
        self.labels = scipy.io.loadmat(row.labels)["label"]
        self.ops = scipy.io.loadmat(row.ops)["op"]
        self.part_mesh_indices = scipy.io.loadmat(row.part_mesh_indices)["cell_boxs_correspond_objSerialNumber"]

        sym_mat = scipy.io.loadmat(row.syms)
        self.syms = sym_mat["sym"]
        self.shapename = str(sym_mat["shapename"].squeeze())

        # Store obj path
        self.models = row.models

        # Store obb path
        self.obbs = row.obbs

In [484]:
# Load CSV and get a Chair

import pandas as pd

# Load dataset
df = pd.read_csv("dataset_overview.csv", index_col="shape")

# Select a random chair
chair_row = df.sample(n=1).iloc[0]

print("Selected chair:", chair_row.name)  # index name


# Chair_840 AMAZING CHAIR

# df = pd.read_csv("dataset_overview.csv", index_col="shape")

# chair_row = df.loc["Chair_3500"]
#chair_row = df.loc["Chair_50"] leg issue
# chair_row = df.loc["Chair_150"]
# chair_row = df.loc["Chair_150"] rot sym

Selected chair: Chair_4218


In [485]:
chair_row

boxes                            dropbox/Chair/boxes/4218.mat
labels                          dropbox/Chair/labels/4218.mat
models                         dropbox/Chair/models/41441.obj
obbs                             dropbox/Chair/obbs/41441.obb
ops                                dropbox/Chair/ops/4218.mat
part_mesh_indices    dropbox/Chair/part mesh indices/4218.mat
syms                              dropbox/Chair/syms/4218.mat
Name: Chair_4218, dtype: object

In [486]:
def display_section(description, data):
    print(f"\n=== {description} ===")
    if isinstance(data, (list, tuple, np.ndarray)):
        print(data)
    else:
        print(f"  {data}")

# Minimal, human-readable descriptions for each folder
file_descriptions = {
    "boxes": "Part bounding boxes",
    "labels": "Semantic labels of parts",
    "models": "3D mesh (.obj)",
    "obbs": "Whole shape bounding box",
    "ops": "Node types (leaf, adjacency, symmetry)",
    "part mesh indices": "Indices of part meshes",
    "syms": "Symmetry parameters"
}

# Example usage for a single shape
chair = ShapeData(chair_row)  # assuming chair_row is a row from your CSV

# Display sections in the requested order
display_section("Chair Shape Name", chair.shapename)
display_section(file_descriptions["boxes"], chair.boxes)
display_section(file_descriptions["labels"], chair.labels)
display_section(file_descriptions["models"], chair.models)
display_section(file_descriptions["obbs"], chair.obbs)
display_section(file_descriptions["ops"], chair.ops)
display_section(file_descriptions["part mesh indices"], chair.part_mesh_indices)
display_section(file_descriptions["syms"], chair.syms)


=== Chair Shape Name ===
  41441

=== Part bounding boxes ===
[[-1.11340e-02 -3.59726e-01 -2.58968e-01  2.36147e-01 -2.20110e-01
  -1.08549e-02 -1.07772e-02]
 [-1.16364e-01 -4.53522e-01 -4.82916e-01 -4.83885e-01 -6.39326e-03
   4.24921e-01  4.21655e-01]
 [ 9.99295e-02  4.34458e-01 -3.01701e-01 -3.00342e-01 -2.43269e-01
  -2.38011e-01 -2.48284e-01]
 [ 1.48594e-01  6.72350e-01  6.29423e-01  6.29642e-01  3.78923e-01
   6.58864e-01  7.43417e-01]
 [ 7.17753e-01  1.05029e-01  6.23205e-02  7.83404e-02  5.14078e-02
   8.68879e-02  7.79208e-02]
 [ 8.16810e-01  3.55070e-02  7.75322e-02  6.67453e-02  8.98005e-02
   5.65700e-01  6.55011e-01]
 [ 0.00000e+00  0.00000e+00  2.37722e-02 -5.80270e-02  1.68422e-01
  -5.23082e-02 -5.23114e-02]
 [ 1.00000e+00  1.00000e+00  9.84941e-01  9.81856e-01  9.84411e-01
   9.93093e-01  9.94333e-01]
 [ 0.00000e+00  0.00000e+00  1.71251e-01  1.80534e-01  5.06833e-02
  -1.05025e-01 -9.25483e-02]
 [ 0.00000e+00  0.00000e+00 -2.43136e-01 -3.43754e-01 -8.85134e-02
   3.0

In [487]:
# My tree looks like this

class Tree(object):
    class NodeType(Enum):
        BOX = 0  # box node
        ADJ = 1  # adjacency (adjacent part assembly) node
        SYM = 2  # symmetry (symmetric part grouping) node

    class Node(object):
        def __init__(self, box=None, left=None, right=None, node_type=None, sym=None, label=None):
            self.box = box          # box feature vector for a leaf node
            self.sym = sym          # symmetry parameter vector for a symmetry node
            self.left = left        # left child for ADJ or SYM (a symmeter generator)
            self.right = right      # right child
            self.node_type = node_type
            self.label = label

        def is_leaf(self):
            return self.node_type == Tree.NodeType.BOX and self.box is not None

        def is_adj(self):
            return self.node_type == Tree.NodeType.ADJ

        def is_sym(self):
            return self.node_type == Tree.NodeType.SYM

    def __init__(self, boxes, ops, syms, labels):
        box_list = [b for b in torch.split(boxes, 1, 0)]
        sym_param = [s for s in torch.split(syms, 1, 0)]
        label_list = [l for l in labels[0]]
        box_list.reverse()
        sym_param.reverse()
        label_list.reverse()
        queue = []
        for id in range(ops.size()[1]):
            if ops[0, id] == Tree.NodeType.BOX.value:
                queue.append(Tree.Node(box=box_list.pop(), node_type=Tree.NodeType.BOX, label=label_list.pop()))
            elif ops[0, id] == Tree.NodeType.ADJ.value:
                left_node = queue.pop()
                right_node = queue.pop()
                queue.append(Tree.Node(left=left_node, right=right_node, node_type=Tree.NodeType.ADJ))
            elif ops[0, id] == Tree.NodeType.SYM.value:
                node = queue.pop()
                queue.append(Tree.Node(left=node, sym=sym_param.pop(), node_type=Tree.NodeType.SYM))
        assert len(queue) == 1
        self.root = queue[0]

boxes = torch.tensor(chair.boxes, dtype=torch.float).t()  # transpose like the GRASSDataset does
ops = torch.tensor(chair.ops, dtype=torch.int)
syms = torch.tensor(chair.syms, dtype=torch.float).t()    # transpose like GRASSDataset
labels = torch.tensor(chair.labels, dtype=torch.int)

# Build the tree
tree = Tree(boxes, ops, syms, labels)

In [488]:
def normalize_vector(vec):
    return vec / np.linalg.norm(vec)

def calculate_orthonormal_basis(dir1, dir2):
    dir1_norm = normalize_vector(dir1)
    dir2_norm = normalize_vector(dir2)
    dir3_norm = normalize_vector(np.cross(dir1_norm, dir2_norm))
    return dir1_norm, dir2_norm, dir3_norm

# Map numeric labels to human-readable part names
label_names = {
    0: "Backrest",
    1: "Seat",
    2: "Leg",
    3: "Armrest"
}

def format_box_as_dict(bv):
    center = np.round(bv[0:3], 7)

    # input order is (len_y, len_z, len_x)
    len_y, len_z, len_x = bv[3], bv[4], bv[5]
    lengths = np.round([len_x, len_y, len_z], 7)  # reorder -> (x, y, z)

    # dir2 = np.round(bv[6:9], 7)
    # dir3 = np.round(bv[9:12], 7)
    # dir1 = np.array([0.0, 0.0, 0.0])              # placeholder

    # get raw dirs
    raw_dir2 = bv[6:9]
    raw_dir3 = bv[9:12]

    # construct orthonormal basis (dir1, dir2, dir3)
    dir2, dir3, dir1  = calculate_orthonormal_basis(raw_dir2, raw_dir3)

    return {
        'center': center,
        'dir1': dir1,
        'dir2': dir2,
        'dir3': dir3,
        'lengths': lengths
    }

def print_box_dict(box, prefix):
    def fmt(arr):
        return "[" + ", ".join(f"{x:.7f}" for x in arr) + "]"
    print(f"{prefix}center : {fmt(box['center'])}")
    print(f"{prefix}dir1   : {fmt(box['dir1'])}")
    print(f"{prefix}dir2   : {fmt(box['dir2'])}")
    print(f"{prefix}dir3   : {fmt(box['dir3'])}")
    print(f"{prefix}lengths: {fmt(box['lengths'])}")

def print_full_tree(node, prefix="", is_last=True):
    branch = "└─ " if is_last else "├─ "

    if node.is_leaf():
        part_name = label_names.get(node.label.item(), f"Unknown({node.label.item()})")
        box_values = node.box.squeeze().tolist()
        box_dict = format_box_as_dict(box_values)
        print(f"{prefix}{branch}Leaf: {part_name}, Label={node.label.item()}")
        print_box_dict(box_dict, prefix + "    ")
    elif node.is_adj():
        print(f"{prefix}{branch}Adjacency Node")
        children = [node.left, node.right]
        for i, child in enumerate(children):
            print_full_tree(child, prefix + ("    " if is_last else "│   "), i == len(children)-1)
    elif node.is_sym():
        sym_values = node.sym.squeeze().tolist()
        print(f"{prefix}{branch}Symmetry Node: Sym Params={[round(x,6) for x in sym_values[:]]}")
        print_full_tree(node.left, prefix + ("    " if is_last else "│   "), True)
    else:
        print(f"{prefix}{branch}Unknown Node Type")

# Print the full tree for your chair
print("Chair Tree:")
print_full_tree(tree.root)

Chair Tree:
└─ Adjacency Node
    ├─ Adjacency Node
    │   ├─ Adjacency Node
    │   │   ├─ Leaf: Backrest, Label=0
    │   │       center : [-0.0107772, 0.4216550, -0.2482840]
    │   │       dir1   : [0.9986308, 0.0520838, -0.0048749]
    │   │       dir2   : [-0.0523114, 0.9943331, -0.0925483]
    │   │       dir3   : [0.0000270, 0.0926766, 0.9956963]
    │   │       lengths: [0.6550110, 0.7434170, 0.0779208]
    │   │   └─ Leaf: Backrest, Label=0
    │   │       center : [-0.0108549, 0.4249210, -0.2380110]
    │   │       dir1   : [0.9986310, 0.0520149, -0.0055319]
    │   │       dir2   : [-0.0523082, 0.9930929, -0.1050250]
    │   │       dir3   : [0.0000308, 0.1051700, 0.9944543]
    │   │       lengths: [0.5657000, 0.6588640, 0.0868879]
    │   └─ Symmetry Node: Sym Params=[0.0, 0.999958, 0.007187, -0.005701, -0.011208, -0.004892, -0.24446, 0.0]
    │       └─ Leaf: Backrest, Label=0
    │           center : [-0.2201100, -0.0063933, -0.2432690]
    │           dir1   : [0.9817

In [489]:
import numpy as np
from scipy.spatial.transform import Rotation

def extract_boxes_and_labels_with_sym_corrected(node):
    """
    Traverse the tree and collect leaf boxes and labels.
    Handles both reflection and rotational symmetry nodes.
    """
    data = {
        'boxes': [],
        'connections': [],
        'symmetries': [],
        'labels': []
    }

    def traverse(n):
        if n.is_leaf():
            box_values = n.box.squeeze().tolist()
            box_dict = format_box_as_dict(box_values)
            box_dict['label_id'] = n.label.item()
            box_dict['label_name'] = label_names.get(n.label.item(), f"Unknown({n.label.item()})")
            
            data['boxes'].append(box_dict)
            data['labels'].append(n.label.item())

        elif n.is_adj():
            # Traverse children normally for adjacency nodes
            if hasattr(n, 'left') and n.left is not None:
                traverse(n.left)
            if hasattr(n, 'right') and n.right is not None:
                traverse(n.right)

        elif n.is_sym():
            sym_values = n.sym.squeeze().tolist()
            sym_type = sym_values[0]
            
            # Collect original boxes from children
            original_boxes = []
            def collect_boxes(sub_node):
                if sub_node.is_leaf():
                    box_values = sub_node.box.squeeze().tolist()
                    box_dict = format_box_as_dict(box_values)
                    box_dict['label_id'] = sub_node.label.item()
                    box_dict['label_name'] = label_names.get(sub_node.label.item(), f"Unknown({sub_node.label.item()})")
                    original_boxes.append(box_dict)
                elif sub_node.is_adj() or sub_node.is_sym():
                    if hasattr(sub_node, 'left') and sub_node.left is not None:
                        collect_boxes(sub_node.left)
                    if hasattr(sub_node, 'right') and sub_node.right is not None:
                        collect_boxes(sub_node.right)
            
            collect_boxes(n)
            
            # Add original boxes to data
            for box in original_boxes:
                data['boxes'].append(box)
                data['labels'].append(box['label_id'])
            
            # Handle different symmetry types
            if sym_type == 0.0:  # Reflection symmetry
                plane_normal = np.array(sym_values[1:4])
                point_on_plane = np.array(sym_values[4:7])
                
                print(f"Reflection symmetry: normal={plane_normal}, point={point_on_plane}")
                
                # Apply reflection transformation to each original box
                for box in original_boxes:
                    reflected_box = box.copy()
                    original_center = box['center']
                    
                    # Reflect point across plane: P' = P - 2 * ((P - P0) · n) * n
                    vector_to_plane = original_center - point_on_plane
                    dot_product = np.dot(vector_to_plane, plane_normal)
                    reflected_center = original_center - 2 * dot_product * plane_normal
                    
                    reflected_box['center'] = np.round(reflected_center, 7)
                    data['boxes'].append(reflected_box)
                    data['labels'].append(reflected_box['label_id'])
            
            elif sym_type == -1.0:  # Rotational symmetry
                rotation_axis = np.array(sym_values[1:4])
                rotation_center = np.array(sym_values[4:7])
                n_fold = int(round(1.0 / sym_values[7]))  # 1/0.25 = 4-fold rotation
                
                print(f"Rotational symmetry: {n_fold}-fold around axis {rotation_axis} at center {rotation_center}")
                
                # Normalize rotation axis
                rotation_axis_norm = rotation_axis / np.linalg.norm(rotation_axis)
                
                # Generate rotated boxes for each rotation angle
                for box in original_boxes:
                    for i in range(1, n_fold):
                        # Calculate rotation angle (in radians)
                        angle = 2 * np.pi * i / n_fold
                        
                        rotated_box = box.copy()
                        original_center = box['center']
                        
                        # Move to origin, rotate, then move back
                        vector_from_center = original_center - rotation_center
                        
                        # Create rotation object
                        rotation = Rotation.from_rotvec(angle * rotation_axis_norm)
                        
                        # Apply rotation
                        rotated_vector = rotation.apply(vector_from_center)
                        rotated_center = rotation_center + rotated_vector
                        
                        # Apply same rotation to orientation vectors
                        rotated_dir1 = rotation.apply(box['dir1'])
                        rotated_dir2 = rotation.apply(box['dir2'])
                        rotated_dir3 = rotation.apply(box['dir3'])
                        
                        rotated_box['center'] = np.round(rotated_center, 7)
                        rotated_box['dir1'] = np.round(rotated_dir1, 7)
                        rotated_box['dir2'] = np.round(rotated_dir2, 7)
                        rotated_box['dir3'] = np.round(rotated_dir3, 7)
                        
                        data['boxes'].append(rotated_box)
                        data['labels'].append(rotated_box['label_id'])
            
            else:
                print(f"Unknown symmetry type: {sym_type}")
                # Fallback: simple mirroring for unknown symmetry types
                for box in original_boxes:
                    reflected_box = box.copy()
                    reflected_box['center'] = np.array([-box['center'][0], box['center'][1], box['center'][2]])
                    data['boxes'].append(reflected_box)
                    data['labels'].append(reflected_box['label_id'])

    traverse(node)
    return data

# Make sure you have these helper functions defined:
def normalize_vector(vec):
    return vec / np.linalg.norm(vec)

def calculate_orthonormal_basis(dir1, dir2):
    dir1_norm = normalize_vector(dir1)
    dir2_norm = normalize_vector(dir2)
    dir3_norm = normalize_vector(np.cross(dir1_norm, dir2_norm))
    return dir1_norm, dir2_norm, dir3_norm

def format_box_as_dict(bv):
    center = np.round(bv[0:3], 7)
    len_y, len_z, len_x = bv[3], bv[4], bv[5]
    lengths = np.round([len_x, len_y, len_z], 7)
    raw_dir2 = bv[6:9]
    raw_dir3 = bv[9:12]
    dir2, dir3, dir1 = calculate_orthonormal_basis(raw_dir2, raw_dir3)
    
    return {
        'center': center,
        'dir1': dir1,
        'dir2': dir2,
        'dir3': dir3,
        'lengths': lengths
    }

# Label mapping
label_names = {
    0: "Backrest",
    1: "Seat", 
    2: "Leg",
    3: "Armrest"
}

# Usage
tree_data = extract_boxes_and_labels_with_sym_corrected(tree.root)
print(f"Total boxes: {len(tree_data['boxes'])}")
print(f"Labels: {tree_data['labels']}")

Reflection symmetry: normal=[ 0.99995798  0.00718682 -0.00570116], point=[-0.0112084  -0.00489185 -0.24446   ]
Reflection symmetry: normal=[ 1. -0. -0.], point=[-0.011133   -0.453522    0.43445799]
Total boxes: 9
Labels: [0, 0, 0, 0, 2, 2, 2, 2, 1]


In [490]:
tree_data

{'boxes': [{'center': array([-0.0107772,  0.421655 , -0.248284 ]),
   'dir1': array([ 0.99863082,  0.05208377, -0.00487492]),
   'dir2': array([-0.05231141,  0.99433311, -0.09254831]),
   'dir3': array([2.70319065e-05, 9.26766257e-02, 9.95696260e-01]),
   'lengths': array([0.655011 , 0.743417 , 0.0779208]),
   'label_id': 0,
   'label_name': 'Backrest'},
  {'center': array([-0.0108549,  0.424921 , -0.238011 ]),
   'dir1': array([ 0.99863099,  0.05201487, -0.00553189]),
   'dir2': array([-0.0523082 ,  0.99309295, -0.10502499]),
   'dir3': array([3.08483066e-05, 1.05170021e-01, 9.94454255e-01]),
   'lengths': array([0.5657   , 0.658864 , 0.0868879]),
   'label_id': 0,
   'label_name': 'Backrest'},
  {'center': array([-0.22011  , -0.0063933, -0.243269 ]),
   'dir1': array([ 0.98173286, -0.17213686,  0.08105247]),
   'dir2': array([0.16842202, 0.98441111, 0.05068331]),
   'dir3': array([-0.08851343, -0.03610651,  0.99542036]),
   'lengths': array([0.0898005, 0.378923 , 0.0514078]),
   'lab

In [491]:
tree_data

{'boxes': [{'center': array([-0.0107772,  0.421655 , -0.248284 ]),
   'dir1': array([ 0.99863082,  0.05208377, -0.00487492]),
   'dir2': array([-0.05231141,  0.99433311, -0.09254831]),
   'dir3': array([2.70319065e-05, 9.26766257e-02, 9.95696260e-01]),
   'lengths': array([0.655011 , 0.743417 , 0.0779208]),
   'label_id': 0,
   'label_name': 'Backrest'},
  {'center': array([-0.0108549,  0.424921 , -0.238011 ]),
   'dir1': array([ 0.99863099,  0.05201487, -0.00553189]),
   'dir2': array([-0.0523082 ,  0.99309295, -0.10502499]),
   'dir3': array([3.08483066e-05, 1.05170021e-01, 9.94454255e-01]),
   'lengths': array([0.5657   , 0.658864 , 0.0868879]),
   'label_id': 0,
   'label_name': 'Backrest'},
  {'center': array([-0.22011  , -0.0063933, -0.243269 ]),
   'dir1': array([ 0.98173286, -0.17213686,  0.08105247]),
   'dir2': array([0.16842202, 0.98441111, 0.05068331]),
   'dir3': array([-0.08851343, -0.03610651,  0.99542036]),
   'lengths': array([0.0898005, 0.378923 , 0.0514078]),
   'lab

In [492]:
import numpy as np
import k3d

# fixed color map for label ids
LABEL_COLORS = {
    0: 0xFF0000,  # red   (Backrest)
    1: 0x00FF00,  # green (Seat)
    2: 0x0000FF,  # blue  (Leg)
    3: 0xFFFF00,  # yellow (Armrest)
}

def plot_tree_data(tree_data):
    """
    Plot all OBB boxes in tree_data as K3D meshes with colors
    according to label_id.
    """
    plot = k3d.plot()

    for box in tree_data['boxes']:
        center = box["center"]
        dir1, dir2, dir3 = box["dir1"], box["dir2"], box["dir3"]
        lengths = box["lengths"]
        label_id = box.get("label_id", -1)

        # 8 corner points
        d1 = 0.5 * lengths[0] * dir1
        d2 = 0.5 * lengths[1] * dir2
        d3 = 0.5 * lengths[2] * dir3

        corners = np.array([
            center - d1 - d2 - d3,
            center - d1 + d2 - d3,
            center + d1 - d2 - d3,
            center + d1 + d2 - d3,
            center - d1 - d2 + d3,
            center - d1 + d2 + d3,
            center + d1 - d2 + d3,
            center + d1 + d2 + d3
        ], dtype=np.float32)

        faces = np.array([
            [0,1,3], [0,3,2],
            [4,6,7], [4,7,5],
            [0,2,6], [0,6,4],
            [1,5,7], [1,7,3],
            [0,4,5], [0,5,1],
            [2,3,7], [2,7,6]
        ], dtype=np.uint32)

        # choose color by label id, fallback to gray
        color = LABEL_COLORS.get(label_id, 0x808080)

        plot += k3d.mesh(corners, faces, color=color)

    plot.display()


# Example usage with tree_data
plot_tree_data(tree_data)

Output()

# OBB SECTION

In [493]:
def print_obb_file(file_path):
    """
    Print the entire contents of a .obb file.
    """
    try:
        with open(file_path, 'r') as f:
            content = f.read()
        print(content)
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

print_obb_file(chair.obbs)

N 9
-0.0108549 0.424921 -0.238011 0.998631 0.0520149 -0.00553192 3.08483e-05 0.10517 0.994454 0.0523082 -0.993093 0.105025 0.5657 0.0868879 0.658864
-0.011134 -0.116364 0.0999295 1 0 0 0 1 0 0 0 1 0.81681 0.148594 0.717753
0.197693 -0.00339045 -0.245651 0.982931 0.180603 0.0350584 -0.0247924 -0.0587903 0.997962 0.182296 -0.981797 -0.0533092 0.0873672 0.0513306 0.372121
-0.22011 -0.00639326 -0.243269 0.168422 0.984411 0.0506833 -0.0885134 -0.0361065 0.99542 0.981733 -0.172136 0.0810525 0.378923 0.0514078 0.0898005
-0.258968 -0.482916 -0.301701 -0.969701 0.0643785 -0.23566 -0.243136 -0.16046 0.956628 0.0237722 0.984941 0.171251 0.0775322 0.0623205 0.629423
0.236147 -0.483885 -0.300342 -0.937265 0.00868856 -0.348509 -0.343754 -0.189431 0.919755 -0.058027 0.981856 0.180534 0.0667453 0.0783404 0.629642
0.33746 -0.453522 0.434458 1 0 0 0 1 0 0 0 1 0.035507 0.67235 0.105029
-0.359726 -0.453522 0.434458 1 0 0 0 1 0 0 0 1 0.035507 0.67235 0.105029
-0.0107772 0.421655 -0.248284 0.998631 0.052083

In [494]:
import numpy as np

def normalize_vector(vec):
    return vec / np.linalg.norm(vec)

def calculate_orthonormal_basis(dir1, dir2):
    dir1_norm = normalize_vector(dir1)
    dir2_norm = normalize_vector(dir2)
    dir3_norm = normalize_vector(np.cross(dir1_norm, dir2_norm))
    return dir1_norm, dir2_norm, dir3_norm

def parse_obb_box_line(vals):
    center = vals[0:3]
    dir1, dir2, dir3 = vals[3:6], vals[6:9], vals[9:12]
    lengths = vals[12:15]
    
    return {
        'center': center,
        'dir1': dir1,
        'dir2': dir2,
        'dir3': dir3,
        'lengths': lengths
    }

def parse_obb_file(file_path):
    """
    Parse an OBB file into dict: boxes, connections, symmetries, labels.
    Each box includes only label_id.
    """
    data = {'boxes': [], 'connections': [], 'symmetries': [], 'labels': []}
    
    with open(file_path, 'r') as f:
        lines = [line.strip() for line in f if line.strip()]

    i = 0
    while i < len(lines):
        line = lines[i]
        parts = line.split()
        if len(parts) < 2:
            i += 1
            continue
            
        section_type = parts[0]
        count = int(parts[1])

        if section_type == 'N':  # boxes
            for _ in range(count):
                i += 1
                vals = np.array(list(map(float, lines[i].split())))
                data['boxes'].append(parse_obb_box_line(vals))

        elif section_type == 'C':  # connections
            for _ in range(count):
                i += 1
                data['connections'].append(list(map(int, lines[i].split())))

        elif section_type == 'S':  # symmetries
            for _ in range(count):
                sym_block = []
                i += 1
                while i < len(lines) and lines[i].split()[0] not in ['N','C','S','L']:
                    sym_block.append(lines[i])
                    i += 1
                i -= 1
                data['symmetries'].append(sym_block)

        elif section_type == 'L':  # labels
            for _ in range(count):
                i += 1
                data['labels'].append(int(lines[i]))
        
        i += 1

    # attach label_id to each box
    for b, lbl in zip(data['boxes'], data['labels']):
        b['label_id'] = lbl

    return data


obb_data = parse_obb_file(chair.obbs)
obb_data

{'boxes': [{'center': array([-0.0108549,  0.424921 , -0.238011 ]),
   'dir1': array([ 0.998631  ,  0.0520149 , -0.00553192]),
   'dir2': array([3.08483e-05, 1.05170e-01, 9.94454e-01]),
   'dir3': array([ 0.0523082, -0.993093 ,  0.105025 ]),
   'lengths': array([0.5657   , 0.0868879, 0.658864 ]),
   'label_id': 0},
  {'center': array([-0.011134 , -0.116364 ,  0.0999295]),
   'dir1': array([1., 0., 0.]),
   'dir2': array([0., 1., 0.]),
   'dir3': array([0., 0., 1.]),
   'lengths': array([0.81681 , 0.148594, 0.717753]),
   'label_id': 1},
  {'center': array([ 0.197693  , -0.00339045, -0.245651  ]),
   'dir1': array([0.982931 , 0.180603 , 0.0350584]),
   'dir2': array([-0.0247924, -0.0587903,  0.997962 ]),
   'dir3': array([ 0.182296 , -0.981797 , -0.0533092]),
   'lengths': array([0.0873672, 0.0513306, 0.372121 ]),
   'label_id': 0},
  {'center': array([-0.22011   , -0.00639326, -0.243269  ]),
   'dir1': array([0.168422 , 0.984411 , 0.0506833]),
   'dir2': array([-0.0885134, -0.0361065,  

In [495]:
import numpy as np
import k3d

# fixed color map for label ids
LABEL_COLORS = {
    0: 0xFF0000,  # red   (Backrest)
    1: 0x00FF00,  # green (Seat)
    2: 0x0000FF,  # blue  (Leg)
    3: 0xFFFF00,  # yellow (Armrest)
}

def plot_obb_data(obb_data):
    """
    Plot all OBB boxes in obb_data as K3D meshes with colors
    according to label_id.
    """
    plot = k3d.plot()

    for box in obb_data['boxes']:
        center = box["center"]
        dir1, dir2, dir3 = box["dir1"], box["dir2"], box["dir3"]
        lengths = box["lengths"]
        label_id = box.get("label_id", -1)

        # 8 corner points
        d1 = 0.5 * lengths[0] * dir1
        d2 = 0.5 * lengths[1] * dir2
        d3 = 0.5 * lengths[2] * dir3

        corners = np.array([
            center - d1 - d2 - d3,
            center - d1 + d2 - d3,
            center + d1 - d2 - d3,
            center + d1 + d2 - d3,
            center - d1 - d2 + d3,
            center - d1 + d2 + d3,
            center + d1 - d2 + d3,
            center + d1 + d2 + d3
        ], dtype=np.float32)

        faces = np.array([
            [0,1,3], [0,3,2],
            [4,6,7], [4,7,5],
            [0,2,6], [0,6,4],
            [1,5,7], [1,7,3],
            [0,4,5], [0,5,1],
            [2,3,7], [2,7,6]
        ], dtype=np.uint32)

        # choose color by label id, fallback to gray
        color = LABEL_COLORS.get(label_id, 0x808080)

        plot += k3d.mesh(corners, faces, color=color)

    plot.display()

# Example usage
plot_obb_data(obb_data)

Output()

# Trimesh

In [496]:
mesh = trimesh.load(chair.models, force='mesh')
# Quick info
print(mesh)  # Shows number of vertices, faces, etc.
# Visualize in a window
mesh.show()

<trimesh.Trimesh(vertices.shape=(22290, 3), faces.shape=(47100, 3))>


# FINAL

In [497]:
plot_tree_data(tree_data)

Output()

In [498]:
plot_obb_data(obb_data)

Output()

In [499]:
mesh.show()